In [187]:
import pandas as pd
import numpy as np

In [188]:
df = pd.read_csv('/content/train.csv')
center = pd.read_csv('/content/fulfilment_center_info.csv')
meal = pd.read_csv('/content/meal_info.csv')

In [189]:
df.head(1)

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177


In [190]:
center.head(1)

,center_id,city_code,region_code,center_type,op_area
0,11,679,56,TYPE_A,3.7


In [191]:
meal.head(1)

,meal_id,category,cuisine
0,1885,Beverages,Thai


In [192]:
df = df.join(
    meal.set_index('meal_id'), on='meal_id').join(center.set_index('center_id'), on='center_id').copy()

In [193]:
df.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


This is used for forecasting demand.

Our FinishedGood(Semiconductor) Table is modeled after the **Kaggle Food Demand Forecasting Dataset**: https://www.kaggle.com/datasets/kannanaikkal/food-demand-forecasting

In [194]:
df = df.rename(
    columns={
        'center_id': 'facility_id',
        'meal_id': 'semiconductor_id',
        'num_orders': 'customer_orders',
        'checkout_price': 'realized_selling_price',
        'base_price': 'list_price'})

In [195]:
# derive capacity before filtering
df["facility_type"] = df["center_type"].map({
    "TYPE_A": "high_capacity",
    "TYPE_B": "mid_capacity",
    "TYPE_C": "low_capacity"
})

capacity_map = {
    "low_capacity": 0.0,
    "mid_capacity": 0.5,
    "high_capacity": 1.0
}

df["facility_capacity_index"] = df["facility_type"].map(capacity_map)

In [196]:
op_area_norm = (
    (df["op_area"] - df["op_area"].min()) /
    (df["op_area"].max() - df["op_area"].min())
)

df["facility_capacity_index"] = (
    0.7 * df["facility_capacity_index"] +
    0.3 * op_area_norm
)

In [197]:
df = df.rename(
    columns={
        'city_code': 'facility_city_id',
        'region_code': 'facility_region_id'})

In [198]:
df.head()

,id,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,category,cuisine,facility_city_id,facility_region_id,center_type,op_area,facility_type,facility_capacity_index
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0,low_capacity,0.054098
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0,low_capacity,0.054098
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0,low_capacity,0.054098
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0,low_capacity,0.054098
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0,low_capacity,0.054098


In [199]:
type_map = {
    "TYPE_A": "high_capacity",
    "TYPE_B": "mid_capacity",
    "TYPE_C": "low_capacity"}

df['facility_type'] = df['center_type'].map(type_map)

In [200]:
df.head()

,id,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,category,cuisine,facility_city_id,facility_region_id,center_type,op_area,facility_type,facility_capacity_index
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0,low_capacity,0.054098
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0,low_capacity,0.054098
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0,low_capacity,0.054098
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0,low_capacity,0.054098
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0,low_capacity,0.054098


# Date Dimension Bridge

week -> approximate date -> month/year

In [201]:
# assume week 1 = Jan 2018 (arbitrary but consistent anchor)
start_date = pd.to_datetime('2023-01-01')

df['date'] = start_date + pd.to_timedelta((df['week'] - 1) * 7, unit='D')

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['year_month'] = df['date'].dt.to_period('M').astype(str)

In [202]:
df

,id,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,category,...,facility_city_id,facility_region_id,center_type,op_area,facility_type,facility_capacity_index,date,year,month,year_month
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456543,1271326,145,61,1543,484.09,484.09,0,0,68,Desert,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10
456544,1062036,145,61,2304,482.09,482.09,0,0,42,Desert,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10
456545,1110849,145,61,2664,237.68,321.07,0,0,501,Salad,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10
456546,1147725,145,61,2569,243.50,313.34,0,0,729,Salad,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10


# Forecasting
We need **Lead-time aware demand coverage**

We are forecasting **demand over supplier lead-time horizon**

For our model: *We forecast demand over the next 16-20 weeks to ensure coverage across all suppliers*

**We train on 135 weeks, validate on last 10 weeks of "training" data**
This proves the model **generalizes to unseen future demand**
 - we validate on holdout future periods to ensure forecasting reliability

 **Procureent decisions are lead-time forward-looking, not short-term reactive**

# Step 1 - Model Validation (for Presentation)
Split:
 - Train: weeks 1-135
 - Validate: weeks 136-145

 Show:
 - error metrics
 - plots
 - model able to generalize to unseen future

 **Credibility in forecasting demand aspect of agent**

In [203]:
df

,id,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,category,...,facility_city_id,facility_region_id,center_type,op_area,facility_type,facility_capacity_index,date,year,month,year_month
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,...,647,56,TYPE_C,2.0,low_capacity,0.054098,2023-01-01,2023,1,2023-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456543,1271326,145,61,1543,484.09,484.09,0,0,68,Desert,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10
456544,1062036,145,61,2304,482.09,482.09,0,0,42,Desert,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10
456545,1110849,145,61,2664,237.68,321.07,0,0,501,Salad,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10
456546,1147725,145,61,2569,243.50,313.34,0,0,729,Salad,...,473,77,TYPE_A,4.5,high_capacity,0.877049,2025-10-05,2025,10,2025-10


In [204]:
df.shape

(456548, 21)

**We have demand history up to early October 2025, and we are now making a procurement decision in December 2025 using the most recent available data**

In real companies:
 - Demand data is lagged
 - Reporting pipelines are not real-time
 - Decisions are made using latest available snapshot

**OUR TIMELINE:**
- Historical demand data:
  - Jan 2023 -> Oct 5 2025 (observed)
- Decision point:
  - Dec 1, 2025 ("today")
- Forecast horizon:
  - Dec 2025 -> Apr 2026 (next 16-20 weeks)

# Fixing Structure of Demand Inventory Data

**Goal we want:**
- Facilities: 2-4
- Semiconductor SKUs: 8-12

In [205]:
df.facility_id.nunique()

77

In [206]:
df.semiconductor_id.nunique()

51

## Selecting/Filtering Top Facilities by volume

In [207]:
facility_volume = df.groupby('facility_id')['customer_orders'].sum()

In [208]:
top_facilities = (facility_volume.sort_values(ascending=False).head(4).index)

In [209]:
df = df[df['facility_id'].isin(top_facilities)]

In [210]:
df.shape

(28024, 21)

## Selecting/Filtering Top Semiconductors by Volume

In [211]:
sku_volume = df.groupby("semiconductor_id").customer_orders.sum()

top_skus = (
    sku_volume.sort_values(ascending=False).head(12).index)

In [212]:
df = df[df['semiconductor_id'].isin(top_skus)]

In [213]:
# reset index
df = df.reset_index(drop=True)

In [214]:
df.shape

(6959, 21)

We've selected PRODUCTS WITH REAL DEMAND and FACILITIES WITH REAL ACTIVITY

No dead SKUs, sparse demand, or noise issues.

**This data will eventually become the fact_semiconductor_demand table**

In [215]:
df.facility_id.nunique()

4

In [216]:
df.semiconductor_id.nunique()

12

Why matters downstream:
1. Forecasting
 - cleaner patterns
 - less noise
 - faster training

2. BOM (Bill of Materials)
 - Manageable mapping
 - interpretable component demand

3. LP Optimization
 - realistic supplier allocation
 - explainable results

# Relabeling SKUs and Facility_IDs

In [217]:
df.semiconductor_id.unique()

array([1885, 1993, 2539, 1778, 1062, 2707, 2290, 1727, 1109, 2826, 1754,
       1971])

In [218]:
df.facility_id.unique()

array([13, 52, 10, 43])

In [219]:
sku_map = {
    old_id: f"SEMICONDUCTOR_{i+1}"
    for i, old_id in enumerate(top_skus)}

facility_map = {
    old_id: f"FACILITY_{i+1}"
    for i, old_id in enumerate(top_facilities)}

df = df.assign(
    semiconductor_id=lambda x: x.semiconductor_id.map(sku_map),
    facility_id=lambda x: x.facility_id.map(facility_map))

In [220]:
df

,id,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,category,...,facility_city_id,facility_region_id,center_type,op_area,facility_type,facility_capacity_index,date,year,month,year_month
0,1171094,1,FACILITY_1,SEMICONDUCTOR_2,135.86,122.28,0,1,2132,Beverages,...,590,56,TYPE_B,6.7,mid_capacity,0.635246,2023-01-01,2023,1,2023-01
1,1068455,1,FACILITY_1,SEMICONDUCTOR_5,134.86,122.28,0,1,2418,Beverages,...,590,56,TYPE_B,6.7,mid_capacity,0.635246,2023-01-01,2023,1,2023-01
2,1105491,1,FACILITY_1,SEMICONDUCTOR_11,133.86,133.86,0,0,474,Beverages,...,590,56,TYPE_B,6.7,mid_capacity,0.635246,2023-01-01,2023,1,2023-01
3,1005971,1,FACILITY_1,SEMICONDUCTOR_12,184.36,182.36,0,0,513,Beverages,...,590,56,TYPE_B,6.7,mid_capacity,0.635246,2023-01-01,2023,1,2023-01
4,1217794,1,FACILITY_1,SEMICONDUCTOR_9,185.33,185.33,0,0,998,Beverages,...,590,56,TYPE_B,6.7,mid_capacity,0.635246,2023-01-01,2023,1,2023-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6954,1092643,145,FACILITY_2,SEMICONDUCTOR_7,464.63,464.63,0,0,555,Rice Bowl,...,590,56,TYPE_A,5.1,high_capacity,0.906557,2025-10-05,2025,10,2025-10
6955,1330371,145,FACILITY_2,SEMICONDUCTOR_3,283.30,283.30,0,0,1796,Rice Bowl,...,590,56,TYPE_A,5.1,high_capacity,0.906557,2025-10-05,2025,10,2025-10
6956,1320167,145,FACILITY_2,SEMICONDUCTOR_8,377.33,379.33,0,0,661,Sandwich,...,590,56,TYPE_A,5.1,high_capacity,0.906557,2025-10-05,2025,10,2025-10
6957,1041395,145,FACILITY_2,SEMICONDUCTOR_6,310.40,312.40,0,0,1147,Sandwich,...,590,56,TYPE_A,5.1,high_capacity,0.906557,2025-10-05,2025,10,2025-10


In [221]:
df.groupby(
    ['facility_id', 'semiconductor_id']
).size().describe()

,0
count,48.000000
mean,144.979167
std,0.144338
min,144.000000
25%,145.000000
50%,145.000000
75%,145.000000
max,145.000000


# Creating Synthetic Features Aligned with Semiconductor Industry

In [222]:
# computing average price per SKU
sku_price = df.groupby("semiconductor_id")["realized_selling_price"].mean()

# ranking into tiers
df["sku_performance_tier"] = pd.qcut(
    df["semiconductor_id"].map(sku_price),
    q=3,
    labels=["low", "mid", "high"]
)

In [223]:
family_map = {
    'low': 'power_management_modules', #basic power ICs and volate regulates
    'mid': 'mixed_signal_interface_modules', # controllers and interface chips
    'high': 'compute_control_modules' # CPUs, GPUs, advanced MCUs
    }

df['finished_family'] = df['sku_performance_tier'].map(family_map)

In [224]:
# rankinging facilities after filtering

facility_volume_filtered = df.groupby("facility_id")["customer_orders"].sum()

df["facility_scale"] = df["facility_id"].map(
    facility_volume_filtered.rank(method="dense", ascending=False)
)

# mapping to labels
scale_map = {
    1: "mega_hub",
    2: "large_dc",
    3: "regional_dc",
    4: "local_dc"
}
df["facility_scale"] = df["facility_scale"].map(scale_map)

In [225]:
# facility demand volatility
facility_volatility = df.groupby("facility_id")["customer_orders"].std()
df["facility_volatility"] = df["facility_id"].map(facility_volatility)

# Attemtping to Scale Price Attributes to bring them up to the range of actual wholesale selling prices for semiconductors

In [226]:
(df.realized_selling_price <= df.list_price).mean()

np.float64(0.7311395315418882)

In [227]:
df.realized_selling_price.describe()

,realized_selling_price
count,6959.000000
mean,237.314916
std,89.198848
min,98.970000
25%,158.110000
50%,224.100000
75%,301.685000
max,466.630000


Something like 2K - 8K (USD) is much more believable for semiconducot components.

In [228]:
# computing scaling factor
current_mean = df.realized_selling_price.mean()
target_mean = 4000

scale_factor = target_mean / current_mean

In [229]:
# applying scaling
df['realized_selling_price'] *= scale_factor
df['list_price'] *= scale_factor

In [230]:
# still holds. CHECK
(df.realized_selling_price <= df.list_price).mean()

np.float64(0.7311395315418882)

Price

In [231]:
df.facility_type.unique()

array(['mid_capacity', 'high_capacity'], dtype=object)

Different finished semiconductor SKUs have different price tiers due to differences in complexity, performance, and quality level. We preserved those relative differences when adapting the demand data.
 - some SKUs are premium / higher-performance chips
 - some are lower-cost / standard chips

 We preserved underlying relative pricing relationships while scaling values into a more realistic semiconductor range.

**WHY ACCEPTABLE**
- We repurposed this demand dataset
- Preserved relative structure instead of inventing random prices
- Project's core value is procurement decision logic, not synthetically deriving down to the dollar exact finished-chip pricing

**DO NOT FORGET**
- User parameters going into scoring (factoring in/how it does):
  - lambda
  - quantity
  - compliance threshold

In [232]:
df.center_type

,center_type
0,TYPE_B
1,TYPE_B
2,TYPE_B
3,TYPE_B
4,TYPE_B
...,...
6954,TYPE_A
6955,TYPE_A
6956,TYPE_A
6957,TYPE_A


In [233]:
df.columns

Index(['id', 'week', 'facility_id', 'semiconductor_id',
       'realized_selling_price', 'list_price', 'emailer_for_promotion',
       'homepage_featured', 'customer_orders', 'category', 'cuisine',
       'facility_city_id', 'facility_region_id', 'center_type', 'op_area',
       'facility_type', 'facility_capacity_index', 'date', 'year', 'month',
       'year_month', 'sku_performance_tier', 'finished_family',
       'facility_scale', 'facility_volatility'],
      dtype='object')

In [234]:
df.drop(columns=[
    'id', 'category', 'cuisine', 'center_type', 'op_area'], inplace=True)

In [235]:
df

,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,facility_city_id,facility_region_id,facility_type,facility_capacity_index,date,year,month,year_month,sku_performance_tier,finished_family,facility_scale,facility_volatility
0,1,FACILITY_1,SEMICONDUCTOR_2,2289.952984,2061.058817,0,1,2132,590,56,mid_capacity,0.635246,2023-01-01,2023,1,2023-01,low,power_management_modules,mega_hub,956.505474
1,1,FACILITY_1,SEMICONDUCTOR_5,2273.097744,2061.058817,0,1,2418,590,56,mid_capacity,0.635246,2023-01-01,2023,1,2023-01,low,power_management_modules,mega_hub,956.505474
2,1,FACILITY_1,SEMICONDUCTOR_11,2256.242503,2256.242503,0,0,474,590,56,mid_capacity,0.635246,2023-01-01,2023,1,2023-01,low,power_management_modules,mega_hub,956.505474
3,1,FACILITY_1,SEMICONDUCTOR_12,3107.432152,3073.721671,0,0,513,590,56,mid_capacity,0.635246,2023-01-01,2023,1,2023-01,low,power_management_modules,mega_hub,956.505474
4,1,FACILITY_1,SEMICONDUCTOR_9,3123.781736,3123.781736,0,0,998,590,56,mid_capacity,0.635246,2023-01-01,2023,1,2023-01,mid,mixed_signal_interface_modules,mega_hub,956.505474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6954,145,FACILITY_2,SEMICONDUCTOR_7,7831.450428,7831.450428,0,0,555,590,56,high_capacity,0.906557,2025-10-05,2025,10,2025-10,high,compute_control_modules,large_dc,1345.705931
6955,145,FACILITY_2,SEMICONDUCTOR_3,4775.089655,4775.089655,0,0,1796,590,56,high_capacity,0.906557,2025-10-05,2025,10,2025-10,mid,mixed_signal_interface_modules,large_dc,1345.705931
6956,145,FACILITY_2,SEMICONDUCTOR_8,6359.987926,6393.698407,0,0,661,590,56,high_capacity,0.906557,2025-10-05,2025,10,2025-10,high,compute_control_modules,large_dc,1345.705931
6957,145,FACILITY_2,SEMICONDUCTOR_6,5231.866674,5265.577155,0,0,1147,590,56,high_capacity,0.906557,2025-10-05,2025,10,2025-10,high,compute_control_modules,large_dc,1345.705931


In [239]:
df = df.rename(
    columns={
        'semiconductor_id': 'finished_sku_id',})


In [240]:
df.to_csv('finished_goods_demand_table.csv', index=False)